<a href="https://colab.research.google.com/github/cruhling289/MSDSCapstone/blob/main/notebooks/MSDSCapstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Creating text variable with the content**

In [46]:
#config.py
k = 2
chunk_size = 1200
overlap = 20
#max_new_tokens=300

In [35]:
!wget -q https://raw.githubusercontent.com/cruhling289/MSDSCapstone/main/data/capstone_day_planning.md
with open("capstone_day_planning.md", "r") as f:
    text1 = f.read()

#print(text[:2000])

**another way to read the text**

In [17]:
# @title
import requests
try:
  rawURL = "https://raw.githubusercontent.com/cruhling289/MSDSCapstone/main/data/capstone_day_planning.md"
  response = requests.get(rawURL, timeout=10)
  response.raise_for_status
  text = response.text
except Exception as e:
  print(f'Unexpected error: {e}')

#print(text)

In [18]:
# @title
import requests

url = "https://datascience.virginia.edu/data-science-capstone/sponsor-experience"

response = requests.get(url)

print(response.status_code)

200


**read text 2**

In [36]:
import requests

try:
  rawURL = "https://datascience.virginia.edu/data-science-capstone/sponsor-experience"
  response = requests.get(rawURL, timeout=10)
  response.raise_for_status()
  web_text = response.text
except Exception as e:
  print(f'Unexpected error: {e}')


from bs4 import BeautifulSoup

soup = BeautifulSoup(web_text, "html.parser")
web_text = soup.get_text(separator="\n")

#print(web_text[:2000])

**Combining the two texts**

In [37]:
text = text1 + "\n\n" + web_text

print(len(text))

13215


**Auto-chunking**

In [38]:
chunkList = []

for i in range(0, len(text), chunk_size - overlap):
    chunk = text[i:i + chunk_size]
    chunkList.append(chunk)

print(len(chunkList))

12


**Manually chunking text, fix later to make it automatic**

In [22]:
# @title
chunks = [
    "## Event and Venue Info",
    "## Preparing for the Event",
    "## Attendance",
    "## Awards",
    "## Presentation Schedule",
]
chunkList = []
currentChunk = ""

for line in text.splitlines():

    if line in chunks:
        if currentChunk != "":
          chunkList.append(currentChunk)
        currentChunk = line + "\n"
    else:
        currentChunk += line + "\n"
chunkList.append(currentChunk)
#print(len(chunkList))



In [39]:
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer


embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [24]:
# @title
print(type(embedding_model))

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


**Embedding chunked text**

In [40]:
embeddings = []

for chunk in chunkList:
    x = embedding_model.encode(chunk)
    embeddings.append(x)

#print(len(embeddings))
#print(len(embeddings[0]))

In [41]:
from sklearn.metrics.pairwise import cosine_similarity

**Vector similarity/retrieval**

In [47]:
query = input("What is your question? ")
queryEmbedding = embedding_model.encode(query)
similarities = []
for embedding in embeddings:
  similarity = cosine_similarity([queryEmbedding], [embedding]) [0][0]
  similarities.append(similarity)
#print((similarities))

top_indices = sorted(
    range(len(similarities)),
    key=lambda i: similarities[i],
    reverse=True
)[:k]

retrievedChunks = []
for i in top_indices:
  retrievedChunks.append(chunkList[i])

What is your question? what are the benefits of sponsoring a capstone


**Combining the k retrieved chunks**

In [48]:
joinedRetrieval = ""
for i in range(len(retrievedChunks)):
  joinedRetrieval += retrievedChunks[i] + "\n\n"


**Generation**

In [49]:
prompt = f'''Using only the following context, answer the question.
  Context: {joinedRetrieval} \n
  Question: {query}\n
  Answer: '''
#print(prompt)

In [30]:
# @title
query = "What are the benefits of sponsoring a capstone?"
queryEmbedding = embedding_model.encode(query)

similarities = []

for embedding in embeddings:
    similarity = cosine_similarity([queryEmbedding], [embedding])
    similarities.append(similarity[0][0])

top_indices = sorted(
    range(len(similarities)),
    key=lambda i: similarities[i],
    reverse=True
)[:3]

for i in top_indices:
    print("CHUNK:", i)
    print("SIMILARITY:", similarities[i])
    print(chunkList[i])
    print("-" * 50)

CHUNK: 0
SIMILARITY: 0.3549329

<img src="https://github.com/UVADS/ds6013/blob/main/capstone_day.png" width="300"/>

#### Last updated: August 10, 2026

#### Residential MSDS Program

---

**PURPOSE:** Details for the presentation event including:

- [Event and Venue Info](#event-and-venue-info)
- [Preparing for the Event](#preparing-for-the-event)
- [Attendance](#attendance)
- [Awards](#awards)
- [Presentation Schedule](#presentation-schedule)


--------------------------------------------------
CHUNK: 3
SIMILARITY: 0.3242929
## Attendance

- We ask that students stay for the full event to support their cohort
- For privacy reasons, we do not record the capstone event 


--------------------------------------------------
CHUNK: 4
SIMILARITY: 0.29273677
## Awards

- Awards will be offered to capstone teams in four broad areas. 

- Awards will be offered on the basis of student and audience voting at the end of the session. (Members of the audience may vote if they have participated in 

**importing the llm (qwen3) through the transformers library**

In [50]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-1.7B", device_map="auto")
messages = [
    {"role": "user", "content": prompt},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
	enable_thinking=False,
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=300)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The benefits of sponsoring a capstone include:

1. **Customized Engagements**: Each partnership is designed to meet a sponsor’s specific objectives. Partners receive actionable recommendations, proof-of-concept tools, and direct exposure to advanced data science and AI methodologies.

2. **Secured Partnerships**: Capstone projects operate under formal agreements that clarify data access, intellectual property, and expectations, while maintaining a flexible structure that accommodates a wide range of sponsor needs.

3. **Robust Solutions**: Students are trained to work with large, complex datasets and deliver solutions that drive decision-making. Project outcomes range from predictive models and analytical frameworks to deployable, web-based applications with measurable impact.

4. **Smooth Handoff to the Sponsor**: The capstone projects are designed to allow for a seamless transition to the sponsor, with clear outcomes and deliverables. Some sponsors have asked student teams to present

In [32]:
print(chunkList[3])

## Attendance

- We ask that students stay for the full event to support their cohort
- For privacy reasons, we do not record the capstone event 


